In [1]:
try:
    spark.stop()
except Exception as e:
    pass

# ⚙️ 1. Setup Spark local (avec support Delta)


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("RecipeSearchPoC")
    # Support Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Broadcast join automatique pour df_index (table d'index souvent < 200 Mo)
    .config("spark.sql.autoBroadcastJoinThreshold", "200m")
    # Gestion de la mémoire occupée par Spark
    .config("spark.executor.memory", "3g")  # Mémoire allouée à chaque nœud de calcul
    .config("spark.driver.memory", "1g")    # Mémoire allouée au chef d'orchestre
    .getOrCreate()
)

print("✅ Spark ready")


✅ Spark ready


# 📥 2. Chargement des données


In [ ]:
DATA_PATH = "../data/recipes_parquets"

# Delta gère nativement le Data Skipping et le Z-Order définis dans le pipeline
df_main      = spark.read.format("delta").load(f"{DATA_PATH}/recipes_main")
df_index     = spark.read.format("delta").load(f"{DATA_PATH}/ingredients_index").cache()
df_nutrition = spark.read.format("delta").load(f"{DATA_PATH}/recipes_nutrition_detail")

df_index.count()

print("Table 'df_index' mise en cache")


26/04/11 10:48:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


# 🔗 3. Vue master


In [ ]:
# 1. On définit la jointure
df_master_logic = df_main.join(df_nutrition, on="recipe_id", how="left")

# 2. On écrit le résultat physiquement dans un nouveau dossier Delta
MASTER_PATH = f"{DATA_PATH}/recipes_master"
df_master_logic.write \
    .format("delta") \
    .mode("overwrite") \
    .save(MASTER_PATH)

# 3. On recharge la table Delta pour la suite du notebook
df_master = spark.read.format("delta").load(MASTER_PATH)

print(f"✅ Vue master matérialisée dans Delta avec {len(df_master.columns)} colonnes")

# 🔎 4. Recherche par nom


In [ ]:
def search_by_name(query: str, limit: int = 20) -> list[dict]:
    rows = (
        df_main
        .filter(col("title").ilike(f"%{query}%"))
        .select("recipe_id", "title", "cook_minutes", "nutri_score", "image_url")
        .limit(limit)
        .collect()
    )
    return [r.asDict() for r in rows]

# Test
search_by_name("pasta")


# 🥕 5. Recherche par ingrédient (le cœur)


In [ ]:
def search_by_ingredient(ingredient):
    """
    df_index est typiquement < 200 Mo → AQE le broadcast automatiquement.
    Le join devient un map-side join, pas de shuffle sur df_master.
    """
    recipe_ids = (
        df_index
        .filter(col("ingredient").ilike(f"%{ingredient}%"))
        .select("recipe_id")
        .distinct()
    )
    return (
        df_master
        .join(recipe_ids, on="recipe_id")
        .select("recipe_id", "title", "cook_minutes", "nutri_score", "image_url", "ingredients_raw")
        .limit(20)
    )

# Test
search_by_ingredient("tomato").toPandas()


# 🧠 6. Recherche avancée (clé du PoC)


In [ ]:
def search_advanced(name=None, ingredient=None, max_time=None, nutri_score=None):
    """
    Les filtres sont poussés avant le join (predicate pushdown).
    Spark n'ouvre que les fichiers Delta concernés.
    """
    df = df_master

    if name:
        df = df.filter(col("title").ilike(f"%{name}%"))

    if max_time:
        df = df.filter(col("cook_minutes") <= max_time)

    if nutri_score:
        df = df.filter(col("nutri_score") == nutri_score)

    # Le join sur df_index vient EN DERNIER pour profiter des filtres précédents
    if ingredient:
        recipe_ids = (
            df_index
            .filter(col("ingredient").ilike(f"%{ingredient}%"))
            .select("recipe_id")
            .distinct()
        )
        df = df.join(recipe_ids, on="recipe_id")

    return df.select(
        "recipe_id",
        "title",
        "cook_minutes",
        "cook_time_category",
        "nutri_score",
        "energy_kcal",
        "image_url",
        "ingredients_raw",
    ).limit(20)

# Test : recettes rapides avec nutri_score A
search_advanced(max_time=30, nutri_score="A").toPandas()


# 🍽️ 7. Affichage recette complète


In [ ]:
def show_recipe(recipe_id):
    """
    .limit(1).collect() = Spark s'arrête dès le 1er match,
    pas de scan complet de la table.
    """
    recipe = (
        df_master
        .filter(col("recipe_id") == recipe_id)
        .limit(1)
        .collect()
    )

    if not recipe:
        print("❌ Recette introuvable")
        return

    r = recipe[0].asDict()

    print("🍽️ ", r.get("title", "—"))
    print("⏱️  Temps      :", r.get("cook_minutes"), "min  [", r.get("cook_time_category"), "]")
    print("🔥 Calories   :", r.get("energy_kcal"), "kcal")
    print("🥗 Nutri-score:", r.get("nutri_score"))
    print("🖼️  Image      :", r.get("image_url"))
    print("\n📝 Ingrédients:")
    print(r.get("ingredients_raw", "—"))
    print("\n👨‍🍳 Instructions:")
    print(r.get("instructions_text", "—"))

# Test : 1er recipe_id disponible
first_id = df_main.select("recipe_id").first()["recipe_id"]
show_recipe(first_id)


# 🛑 8. Arrêt Spark


In [ ]:
spark.stop()
print("✅ Spark stopped")